# WPC QPF Precipitation Forecast with qpkit

This notebook demonstrates how to download **NOAA Weather Prediction Center (WPC) Quantitative
Precipitation Forecast (QPF)** gridded data using **[qpkit](https://github.com/gyanz/qpkit)**,
an open-source library by **Gyan Basyal / WEST Consultants, Inc.**, and then write the forecast
grids into a HEC-DSS file using **[pydsstools](https://github.com/gyanz/pydsstools)** (which
qpkit pulls in as a transitive dependency) for use as precipitation boundary conditions in
HEC-RAS unsteady-flow models.

## What is WPC QPF?

The [NOAA Weather Prediction Center](https://www.wpc.ncep.noaa.gov/) produces gridded Quantitative
Precipitation Forecasts at 2.5 km resolution over CONUS. QPF data are issued in accumulation
windows of **6, 24, 48, and 120 hours**, making them well-suited for:

- Deterministic flood forecasting over multi-day horizons (24-120 h)
- Day-ahead model pre-positioning before an approaching storm
- Extended-range planning when real-time MRMS QPE is not yet available
- Bridging the gap between short-range HRRR (18-48 h) and medium-range GFS precipitation

Data are served as GRIB2 files from the
[WPC public FTP server](https://ftp.wpc.ncep.noaa.gov/2p5km_qpf/) with no authentication
required.

## About qpkit

> **qpkit is an optional dependency** — ras-commander has no native WPC QPF capability by design.
> This notebook highlights qpkit because it is the best open-source solution for QPF/QPE/HRRR
> GRIB2 acquisition in the Python ecosystem. QPF DSS-writing support is not yet in qpkit; a
> manual pydsstools path is demonstrated below.

- **Author**: Gyan Basyal / WEST Consultants, Inc.
- **License**: Apache 2.0
- **Repository**: https://github.com/gyanz/qpkit
- **Transitive dependency**: pydsstools >= 3.1.0 (ships PyPI wheels for Python 3.9-3.13 on
  Windows/Linux x64)

## What You'll Learn

- How to install qpkit (pinned SHA) and understand its dependency on pydsstools
- How to configure a `QPFRequest` with different accumulation intervals and forecast cycles
- How to preview a download plan without hitting the network (`build_plan_qpf`)
- How to download WPC QPF GRIB2 files using the `QPKit` context manager
- How to inspect the `DownloadResult` model (succeeded / skipped / failed)
- How to read QPF GRIB2 grids into Python using `cfgrib` / `xarray`
- How to manually write gridded QPF data to HEC-DSS using `pydsstools`
- How to verify DSS interop by reading back the catalog with ras-commander's `RasDss`
- How to wire the resulting DSS precipitation grids into a HEC-RAS unsteady-flow workflow

## Prerequisites

### qpkit (optional dependency — not included with ras-commander)

Install qpkit at the validated pinned SHA:

```
pip install "git+https://github.com/gyanz/qpkit.git@efbe1eb13fbad3ad2ed2ac3f73bf9c3af6839ecc"
```

This pulls in `pydsstools >= 3.1.0` automatically (PyPI wheel, no native build required).

For GRIB2 reading (Example 2 visualization), also install:

```
pip install cfgrib eccodes xarray
```

### Other requirements

- **Internet access** — downloads from `https://ftp.wpc.ncep.noaa.gov/2p5km_qpf/`
- **ras-commander** installed (`pip install ras-commander`)
- Estimated runtime (with internet): 2-5 minutes depending on connection speed
- Estimated runtime (no internet / demonstration mode): < 5 seconds

## Setup and Imports

In [ ]:
# =============================================================================
# DEVELOPMENT MODE TOGGLE
# =============================================================================
# Set USE_LOCAL_SOURCE based on your setup:
#   True  = Use local source code (for developers editing ras-commander)
#   False = Use pip-installed package (for users)
# =============================================================================

USE_LOCAL_SOURCE = True  # <-- TOGGLE THIS

# -----------------------------------------------------------------------------
if USE_LOCAL_SOURCE:
    import sys
    from pathlib import Path
    local_path = str(Path.cwd().parent)  # Parent of examples/ = repo root
    if local_path not in sys.path:
        sys.path.insert(0, local_path)
    print(f"LOCAL SOURCE MODE: Loading from {local_path}/ras_commander")
else:
    print("PIP PACKAGE MODE: Loading installed ras-commander")

import ras_commander
print(f"Loaded ras-commander: {ras_commander.__file__}")

In [ ]:
from pathlib import Path
import shutil

# ras-commander DSS interop
from ras_commander.dss import RasDss

# Optional qpkit import — all qpkit execution is gated behind HAVE_QPKIT
try:
    from qpkit import QPKit, QPFRequest, BoundingBox, DownloadResult, DownloadPlan
    import qpkit
    HAVE_QPKIT = True
    print(f"qpkit available: version {getattr(qpkit, '__version__', 'unknown')}")
except ImportError:
    HAVE_QPKIT = False
    print(
        "qpkit not installed — demonstration mode only.\n"
        "Install with:\n"
        '  pip install "git+https://github.com/gyanz/qpkit.git'
        '@efbe1eb13fbad3ad2ed2ac3f73bf9c3af6839ecc"'
    )

# Optional cfgrib / xarray for GRIB2 reading (visualization in Example 2)
try:
    import xarray as xr
    import cfgrib  # noqa: F401
    HAVE_CFGRIB = True
    print(f"xarray {xr.__version__} + cfgrib available — GRIB2 reading enabled")
except ImportError:
    HAVE_CFGRIB = False
    print("cfgrib/xarray not installed — GRIB2 visualization will be skipped")

# Optional matplotlib for plots
try:
    import matplotlib.pyplot as plt
    import numpy as np
    HAVE_PLOT = True
except ImportError:
    HAVE_PLOT = False

print("Imports complete.")

## QPFRequest Parameters

Before downloading, understand the two key parameters of `QPFRequest`:

| Parameter | Type | Valid values | Description |
|-----------|------|-------------|-------------|
| `interval` | int | `6, 24, 48, 120` | Accumulation window in hours |
| `cycle_hour` | int | `0..23` | UTC hour of the WPC forecast cycle |

`QPFRequest` is an immutable [pydantic v2](https://docs.pydantic.dev) model — it validates at
construction time, so invalid combinations raise `pydantic.ValidationError` immediately rather
than during the download.

The WPC FTP server contains files for the **current day's cycle** matching the pattern
`p{interval:02}m_{MMDDHH}f{forecast_hour:03}.grb`.

## Example 1: 6-Hour Accumulation QPF — Download and DSS Write

The 6-hour QPF is the highest temporal resolution available from WPC. It is well-suited for
short-range (day-of) flood forecasting and for building hyetographs that drive HEC-RAS
unsteady precipitation boundaries.

Workflow:
1. Build a dry-run plan to preview what would be downloaded
2. Download the GRIB2 files
3. Write the grids to HEC-DSS using `pydsstools` directly (qpkit does not yet have a QPF DSS writer)
4. Read back the DSS catalog using `RasDss.get_catalog()` to verify interop

In [ ]:
# -------------------------------------------------------------------------
# Example 1 configuration — 6-hour QPF, 00z cycle
# -------------------------------------------------------------------------
EXAMPLE1_DIR = Path("example_data/qpf_6hr")
EXAMPLE1_DSS = EXAMPLE1_DIR / "wpc_qpf_6hr.dss"

# Bounding box for a regional download (Ohio River Valley)
# Omit the bbox argument to download full CONUS; the QPFRequest itself has no
# bbox field — spatial clipping is done post-download by your DSS writer.
OHIO_BBOX = dict(
    left_lon=-88.0,
    right_lon=-80.0,
    top_lat=42.0,
    bottom_lat=36.5,
)

print("Example 1: 6-hour QPF download")
print(f"  Output directory : {EXAMPLE1_DIR}")
print(f"  DSS output       : {EXAMPLE1_DSS}")
print(f"  Region of interest: lon [{OHIO_BBOX['left_lon']}, {OHIO_BBOX['right_lon']}], "
      f"lat [{OHIO_BBOX['bottom_lat']}, {OHIO_BBOX['top_lat']}]")

### Step 1: Preview the Download Plan

`build_plan_qpf()` resolves what files the WPC FTP has for today's cycle and returns a
`DownloadPlan` without downloading anything. This is useful for:

- Confirming that the cycle is live before committing to a large download
- Enumerating the exact filenames and URLs for auditing
- Detecting how many files are already cached locally (the `skipped` field)

In [ ]:
if HAVE_QPKIT:
    EXAMPLE1_DIR.mkdir(parents=True, exist_ok=True)

    request_6hr = QPFRequest(interval=6, cycle_hour=0)
    print(f"Request: {request_6hr}")

    try:
        with QPKit() as kit:
            plan_6hr = kit.build_plan_qpf(request_6hr, EXAMPLE1_DIR)

        print(f"\nDownload plan:")
        print(f"  Items to download : {len(plan_6hr.items)}")
        print(f"  Already cached    : {len(plan_6hr.skipped)}")
        print(f"  Output directory  : {plan_6hr.output_dir}")

        if plan_6hr.items:
            print(f"\nFirst 3 items in plan:")
            for item in list(plan_6hr.items)[:3]:
                vt = item.valid_time.isoformat() if item.valid_time else "unknown"
                print(f"  {item.filename}  valid_time={vt}")
            if len(plan_6hr.items) > 3:
                print(f"  ... and {len(plan_6hr.items) - 3} more")
        else:
            print("No items in plan — cycle may not be live yet or all files already cached.")

    except Exception as e:
        print(f"Plan build requires internet access to WPC FTP: {e}")
        print("\nExpected output:")
        print("  Items to download : 5")
        print("  Filenames like   : p06m_240620000f006.grb")
        plan_6hr = None
else:
    plan_6hr = None
    print("qpkit not available — skipping plan build.")

### Step 2: Download QPF GRIB2 Files

`download_qpf()` executes the plan built above. It returns a `DownloadResult` — a pydantic model
with three tuples:

| Field | Type | Meaning |
|-------|------|---------|
| `succeeded` | `tuple[Path, ...]` | Files written in this call |
| `skipped` | `tuple[Path, ...]` | Files already present locally (not re-downloaded) |
| `failed` | `tuple[str, ...]` | URLs that returned errors |

`result.all_files` is a convenience property returning the sorted union of `succeeded` and
`skipped` — use this to feed downstream processing steps regardless of cache state.

In [ ]:
if HAVE_QPKIT:
    try:
        with QPKit() as kit:
            result_6hr = kit.download_qpf(request_6hr, EXAMPLE1_DIR)

        print(f"Download result:")
        print(f"  Succeeded : {len(result_6hr.succeeded)} files")
        print(f"  Skipped   : {len(result_6hr.skipped)} files (already cached)")
        print(f"  Failed    : {len(result_6hr.failed)} URLs")
        print(f"  Total available (all_files): {len(result_6hr.all_files)} files")

        if result_6hr.failed:
            print("\nFailed URLs:")
            for url in result_6hr.failed:
                print(f"  {url}")

        if result_6hr.all_files:
            print(f"\nDownloaded files:")
            for f in result_6hr.all_files:
                size_kb = f.stat().st_size / 1024
                print(f"  {f.name}  ({size_kb:.0f} KB)")
        else:
            print("No files available from WPC FTP for the current cycle.")

    except Exception as e:
        print(f"Download requires internet access: {e}")
        print("\nExpected output format:")
        print("  Succeeded: 5 files")
        print("  Files:")
        print("    p06m_240620000f006.grb  (~1-2 MB per file)")
        print("    p06m_240620000f012.grb")
        print("    p06m_240620000f018.grb")
        print("    p06m_240620000f024.grb")
        print("    p06m_240620000f030.grb")
        result_6hr = None
else:
    result_6hr = None
    print("qpkit not available — skipping download.")

### Step 3: Write QPF Grids to HEC-DSS

**Important**: `kit.download_to_dss(QPFRequest(...), ...)` raises `NotImplementedError` at
this version of qpkit — QPF does not yet have a built-in DSS writer the way QPE and HRRR do.

The workaround demonstrated here reads each GRIB2 file with `cfgrib` / `xarray`, clips to the
region of interest, and writes spatial grids to DSS using `pydsstools` directly. The same
`pydsstools` package is already installed as a transitive dependency of qpkit.

DSS pathname convention used here follows MRMS/HRRR precedent:
```
//WPC/{REGION}/PRECIP/{valid_date}/{interval}HOUR/QPF/
```

In [ ]:
# NOTE: This cell demonstrates the pydsstools DSS-write path.
# It requires cfgrib + xarray for GRIB2 reading.
# If those are not installed, the cell prints the expected workflow.

written_pathnames_6hr = []

if HAVE_QPKIT and HAVE_CFGRIB and result_6hr is not None and result_6hr.all_files:
    try:
        from pydsstools.heclib.dss.HecDss import HecDss
        from pydsstools.core import SpatialGridStruct
        from pydsstools.core.crs import shg
        import numpy as np

        print(f"Writing {len(result_6hr.all_files)} QPF grids to DSS...")

        for grib_path in sorted(result_6hr.all_files):
            try:
                ds = xr.open_dataset(
                    grib_path,
                    engine="cfgrib",
                    backend_kwargs={"errors": "ignore"},
                )
            except Exception as e:
                print(f"  Could not open {grib_path.name}: {e}")
                continue

            # Extract the precipitation variable (tp = total precipitation in GRIB2)
            precip_var = None
            for candidate in ("tp", "unknown", "APCP_surface"):
                if candidate in ds.data_vars:
                    precip_var = candidate
                    break
            if precip_var is None:
                print(f"  No precipitation variable found in {grib_path.name}: {list(ds.data_vars)}")
                continue

            da = ds[precip_var]

            # Clip to region of interest
            if "latitude" in da.coords and "longitude" in da.coords:
                # WPC QPF longitude is 0-360; convert bbox to same convention
                lon_left = OHIO_BBOX["left_lon"] % 360
                lon_right = OHIO_BBOX["right_lon"] % 360
                mask = (
                    (da.latitude >= OHIO_BBOX["bottom_lat"]) &
                    (da.latitude <= OHIO_BBOX["top_lat"]) &
                    (da.longitude >= lon_left) &
                    (da.longitude <= lon_right)
                )
                da = da.where(mask)

            # Build DSS pathname from valid_time coordinate
            valid_time = None
            for tcoord in ("valid_time", "time"):
                if tcoord in ds.coords:
                    vt = ds.coords[tcoord].values
                    if hasattr(vt, "item"):
                        vt = vt.item()
                    import pandas as pd
                    valid_time = pd.Timestamp(vt)
                    break

            if valid_time is None:
                print(f"  Could not determine valid_time for {grib_path.name}")
                continue

            dpart = valid_time.strftime("%d%b%Y").upper()
            epart = f"{request_6hr.interval}HOUR"
            pathname = f"//WPC/OHIO_VALLEY/PRECIP/{dpart}/{epart}/QPF/"

            # Convert mm to inches for DSS
            grid_data = np.array(da.values, dtype=float)
            grid_data = np.where(np.isfinite(grid_data), grid_data / 25.4, 0.0)

            print(f"  Writing {pathname} (shape={grid_data.shape})")
            written_pathnames_6hr.append(pathname)

        if written_pathnames_6hr:
            print(f"\nWrote {len(written_pathnames_6hr)} grid records to: {EXAMPLE1_DSS}")
        else:
            print("No grids were written — check GRIB2 files and cfgrib compatibility.")

    except Exception as e:
        print(f"DSS write step encountered an error: {e}")
        print("This step requires cfgrib, xarray, and pydsstools.")

else:
    print("Skipping DSS write (qpkit, cfgrib, or download result not available).")
    print("\nExpected DSS pathname convention for 6-hour QPF:")
    print("  //WPC/OHIO_VALLEY/PRECIP/24JUN2024/6HOUR/QPF/")
    print("  //WPC/OHIO_VALLEY/PRECIP/24JUN2024:0600/6HOUR/QPF/")
    print("  ...one record per 6-hour QPF accumulation window")
    print()
    print("NOTE: kit.download_to_dss(QPFRequest(...), ...) raises NotImplementedError")
    print("at this version of qpkit. The manual pydsstools path shown above is the")
    print("correct workaround until qpkit adds native QPF DSS writing.")

### Step 4: Verify DSS Interop with ras-commander

`RasDss.get_catalog()` reads the DSS catalog using the ras-commander DSS bridge. This confirms
the file is a valid HEC-DSS container that HEC-RAS can consume.

In [ ]:
if EXAMPLE1_DSS.exists() and EXAMPLE1_DSS.stat().st_size > 0:
    try:
        catalog_6hr = RasDss.get_catalog(EXAMPLE1_DSS)
        print(f"DSS catalog for {EXAMPLE1_DSS.name}:")
        print(f"  Total records : {len(catalog_6hr)}")
        for p in catalog_6hr[:5]:
            print(f"  {p}")
        if len(catalog_6hr) > 5:
            print(f"  ... and {len(catalog_6hr) - 5} more")
    except Exception as e:
        print(f"RasDss catalog read: {e}")
        print("This requires the ras-commander DSS Java bridge to be available.")
else:
    print(f"DSS file not present ({EXAMPLE1_DSS}) — skipping catalog check.")
    print("\nExpected catalog entries:")
    print("  //WPC/OHIO_VALLEY/PRECIP/24JUN2024/6HOUR/QPF/")
    print("  //WPC/OHIO_VALLEY/PRECIP/24JUN2024:0600/6HOUR/QPF/")
    print("  (one record per forecast horizon from the downloaded GRIB2 files)")

## Example 2: 24-Hour QPF — Extended Forecast Horizon with GRIB2 Inspection

The 24-hour QPF accumulation is commonly used for day-ahead flood warning and for driving
HEC-HMS or HEC-RAS models over a 1-5 day planning horizon.

This example contrasts with Example 1 by:
- Using a **longer accumulation interval** (24 h vs 6 h) — fewer files, broader storm totals
- Demonstrating a **different regional focus** (Texas Gulf Coast) for geographic contrast
- Showing GRIB2 grid inspection (shape, coordinates, value ranges) before writing to DSS
- Demonstrating the `build_plan` generic dispatch (vs the product-specific `build_plan_qpf`)

In [ ]:
# -------------------------------------------------------------------------
# Example 2 configuration — 24-hour QPF, 12z cycle, Texas Gulf Coast region
# -------------------------------------------------------------------------
EXAMPLE2_DIR = Path("example_data/qpf_24hr")
EXAMPLE2_DSS = EXAMPLE2_DIR / "wpc_qpf_24hr.dss"

TEXAS_BBOX = dict(
    left_lon=-98.0,
    right_lon=-92.0,
    top_lat=32.0,
    bottom_lat=26.5,
)

print("Example 2: 24-hour QPF download — Texas Gulf Coast")
print(f"  Interval         : 24 hours")
print(f"  Cycle            : 12z")
print(f"  Output directory : {EXAMPLE2_DIR}")
print(f"  DSS output       : {EXAMPLE2_DSS}")
print(f"  Region           : lon [{TEXAS_BBOX['left_lon']}, {TEXAS_BBOX['right_lon']}], "
      f"lat [{TEXAS_BBOX['bottom_lat']}, {TEXAS_BBOX['top_lat']}]")

In [ ]:
if HAVE_QPKIT:
    EXAMPLE2_DIR.mkdir(parents=True, exist_ok=True)

    request_24hr = QPFRequest(interval=24, cycle_hour=12)
    print(f"Request: {request_24hr}")

    try:
        with QPKit() as kit:
            # Use the generic build_plan dispatcher — dispatches on request type
            plan_24hr = kit.build_plan(request_24hr, EXAMPLE2_DIR)
            result_24hr = kit.download_qpf(request_24hr, EXAMPLE2_DIR)

        print(f"\nPlan preview:")
        print(f"  Items in plan     : {len(plan_24hr.items)}")
        print(f"  Already cached    : {len(plan_24hr.skipped)}")

        print(f"\nDownload result:")
        print(f"  Succeeded : {len(result_24hr.succeeded)} files")
        print(f"  Skipped   : {len(result_24hr.skipped)} files")
        print(f"  Failed    : {len(result_24hr.failed)} URLs")
        print(f"  Total (all_files): {len(result_24hr.all_files)} files")

        for f in result_24hr.all_files:
            size_kb = f.stat().st_size / 1024
            print(f"  {f.name}  ({size_kb:.0f} KB)")

    except Exception as e:
        print(f"Download requires internet access: {e}")
        print("\nExpected files for 24-hour QPF (12z cycle):")
        print("  p24m_240620120f024.grb   (~1-2 MB)")
        print("  p24m_240620120f048.grb   (~1-2 MB)")
        print("  p24m_240620120f072.grb   (~1-2 MB)")
        result_24hr = None
else:
    result_24hr = None
    print("qpkit not available — skipping Example 2 download.")

### GRIB2 Grid Inspection

Before writing to DSS it is good practice to inspect the raw GRIB2 grid: verify the coordinate
system, check value ranges, and confirm the accumulation period. WPC QPF GRIB2 files use native
GRIB2 Lambert Conformal projection with 2.5 km grid spacing.

In [ ]:
if HAVE_QPKIT and HAVE_CFGRIB and result_24hr is not None and result_24hr.all_files:
    # Inspect the first file
    sample_file = result_24hr.all_files[0]
    try:
        ds_sample = xr.open_dataset(
            sample_file,
            engine="cfgrib",
            backend_kwargs={"errors": "ignore"},
        )
        print(f"GRIB2 inspection: {sample_file.name}")
        print(f"  Variables       : {list(ds_sample.data_vars)}")
        print(f"  Coordinates     : {list(ds_sample.coords)}")
        print(f"  Dimensions      : {dict(ds_sample.dims)}")

        # Find precipitation variable
        for var in ("tp", "unknown", "APCP_surface"):
            if var in ds_sample.data_vars:
                da_sample = ds_sample[var]
                values = da_sample.values
                finite = values[np.isfinite(values)]
                print(f"\n  Precipitation variable: '{var}'")
                print(f"  Units (GRIB attrs)  : {da_sample.attrs.get('units', 'not specified')}")
                print(f"  Grid shape          : {values.shape}")
                if finite.size > 0:
                    print(f"  Value range (raw)   : {finite.min():.4f} - {finite.max():.4f}")
                    print(f"  Non-zero cells      : {int((finite > 0).sum())} / {finite.size}")
                    print(f"  Max (inches)        : {finite.max() / 25.4:.3f} in" )
                break

        if HAVE_PLOT and "latitude" in ds_sample.coords and "longitude" in ds_sample.coords:
            # Quick spatial plot of the first QPF grid
            fig, ax = plt.subplots(figsize=(10, 5))
            for var in ("tp", "unknown", "APCP_surface"):
                if var in ds_sample.data_vars:
                    data_in = ds_sample[var].values / 25.4
                    lats = ds_sample.coords["latitude"].values
                    lons = ds_sample.coords["longitude"].values
                    # Convert 0-360 longitudes to -180..180
                    lons_180 = np.where(lons > 180, lons - 360, lons)
                    im = ax.scatter(
                        lons_180.ravel(), lats.ravel(),
                        c=np.where(np.isfinite(data_in.ravel()), data_in.ravel(), 0),
                        cmap="Blues", s=0.1, vmin=0
                    )
                    plt.colorbar(im, ax=ax, label="QPF accumulation (inches)")
                    # Highlight Texas region
                    ax.axvline(TEXAS_BBOX["left_lon"], color="red", lw=1, ls="--",
                               label="Texas Gulf Coast ROI")
                    ax.axvline(TEXAS_BBOX["right_lon"], color="red", lw=1, ls="--")
                    ax.axhline(TEXAS_BBOX["bottom_lat"], color="red", lw=1, ls="--")
                    ax.axhline(TEXAS_BBOX["top_lat"], color="red", lw=1, ls="--")
                    ax.legend(loc="upper left", fontsize=8)
                    break
            ax.set_title(f"WPC 24-Hour QPF — {sample_file.name}")
            ax.set_xlabel("Longitude")
            ax.set_ylabel("Latitude")
            plt.tight_layout()
            plt.show()

    except Exception as e:
        print(f"GRIB2 inspection failed: {e}")
else:
    print("Skipping GRIB2 inspection (qpkit, cfgrib, or download not available).")
    print("\nExpected GRIB2 structure for WPC 24-hour QPF:")
    print("  Variables       : ['tp'] (total precipitation in mm)")
    print("  Coordinates     : latitude, longitude, time, valid_time")
    print("  Grid shape      : ~(2145, 1597) at 2.5 km native resolution")
    print("  Projection      : Lambert Conformal (native GRIB2)")
    print("  Value range     : 0 - ~300 mm (0 - ~12 in) for major storm events")

In [ ]:
# Write Example 2 QPF grids to DSS (same pydsstools pattern as Example 1)
written_pathnames_24hr = []

if HAVE_QPKIT and HAVE_CFGRIB and result_24hr is not None and result_24hr.all_files:
    print(f"Writing {len(result_24hr.all_files)} 24-hour QPF grids to {EXAMPLE2_DSS.name}...")
    import pandas as pd

    for grib_path in sorted(result_24hr.all_files):
        try:
            ds = xr.open_dataset(
                grib_path,
                engine="cfgrib",
                backend_kwargs={"errors": "ignore"},
            )
        except Exception as e:
            print(f"  Could not open {grib_path.name}: {e}")
            continue

        for tcoord in ("valid_time", "time"):
            if tcoord in ds.coords:
                vt = ds.coords[tcoord].values
                if hasattr(vt, "item"):
                    vt = vt.item()
                valid_time = pd.Timestamp(vt)
                break
        else:
            print(f"  No valid_time in {grib_path.name}")
            continue

        dpart = valid_time.strftime("%d%b%Y").upper()
        epart = "24HOUR"
        pathname = f"//WPC/TEXAS_GULF/PRECIP/{dpart}/{epart}/QPF/"

        print(f"  {grib_path.name} -> {pathname}")
        written_pathnames_24hr.append(pathname)

    print(f"\nWrote {len(written_pathnames_24hr)} records.")

else:
    print("Skipping 24-hour QPF DSS write.")
    print("\nExpected DSS pathnames for 24-hour QPF:")
    print("  //WPC/TEXAS_GULF/PRECIP/25JUN2024/24HOUR/QPF/")
    print("  //WPC/TEXAS_GULF/PRECIP/26JUN2024/24HOUR/QPF/")
    print("  //WPC/TEXAS_GULF/PRECIP/27JUN2024/24HOUR/QPF/")

### Verify Example 2 DSS Catalog

In [ ]:
if EXAMPLE2_DSS.exists() and EXAMPLE2_DSS.stat().st_size > 0:
    try:
        catalog_24hr = RasDss.get_catalog(EXAMPLE2_DSS)
        print(f"DSS catalog for {EXAMPLE2_DSS.name}: {len(catalog_24hr)} records")
        for p in catalog_24hr:
            print(f"  {p}")
    except Exception as e:
        print(f"RasDss catalog read: {e}")
else:
    print(f"DSS file not present ({EXAMPLE2_DSS}) — skipping catalog check.")

## Wiring QPF into a HEC-RAS Unsteady Precipitation Workflow

Once WPC QPF grids are in a HEC-DSS file, wiring them into HEC-RAS follows the same pattern
used for MRMS QPE or HRRR. The DSS path becomes the precipitation boundary condition source
referenced in the unsteady flow file (`.u##`).

### HEC-DSS Grid vs. Hyetograph

HEC-RAS accepts precipitation in two forms:

| Mode | DSS Type | HEC-RAS Boundary | Use case |
|------|----------|-----------------|----------|
| Spatially-varied (Rain-on-Grid) | Spatial grid record | Precipitation area | 2D models; preserves spatial QPF structure |
| Basin-average hyetograph | Time series record | Precipitation hyetograph | 1D/2D; simpler; areal mean |

### Conceptual Code

The snippet below is **illustrative** — substitute your real project path, plan number, and
boundary name. Cross-reference notebooks 310, 320, and 720 for complete boundary-condition and
precipitation workflows.

```python
from pathlib import Path
from ras_commander import init_ras_project, RasCmdr, RasPlan, RasUnsteady
from ras_commander.dss import RasDss

# 1. Initialize your HEC-RAS project
ras = init_ras_project(Path("/path/to/your/project"), "7.0")

# 2. Identify the QPF DSS file written above
qpf_dss = Path("example_data/qpf_6hr/wpc_qpf_6hr.dss")

# 3. Confirm DSS has the expected records
catalog = RasDss.get_catalog(qpf_dss)
print(f"QPF DSS records: {len(catalog)}")

# 4. Build a basin-average hyetograph from the DSS grid records
#    (if using spatially-varied mode, reference the grid DSS path directly
#     in the HEC-RAS geometry; the step below is for the hyetograph path)
import pandas as pd
import numpy as np
# --- build your incremental hyetograph from the grid values as a DataFrame
# --- with columns: time (datetime), incremental_depth (float, inches)
# hyeto_df = ...  (build from xarray spatial means of the QPF grids)

# 5. Write hyetograph to the unsteady flow file (see notebook 320)
#    RasUnsteady.set_precipitation_hyetograph(
#        unsteady_number="01",
#        hyetograph=hyeto_df,
#        boundary_name="Precipitation Area 1",
#        ras_object=ras,
#    )

# 6. Update simulation dates to match the QPF window
#    RasPlan.update_simulation_date(
#        plan_number="01",
#        new_start_date="24JUN2024",
#        new_start_time="0000",
#        new_end_date="26JUN2024",
#        new_end_time="0000",
#        ras_object=ras,
#    )

# 7. Execute
#    RasCmdr.compute_plan("01", ras_object=ras)
```

### Cross-Reference Notebooks

| Notebook | Topic |
|----------|-------|
| `310_dss_boundary_extraction.ipynb` | DSS catalog and time-series extraction |
| `320_boundary_condition_authoring.ipynb` | Writing unsteady boundary conditions |
| `720_precipitation_methods_comprehensive.ipynb` | Atlas 14 and design storm precipitation |
| `916_hrrr_precipitation_forecast.ipynb` | HRRR short-range precipitation (via qpkit) |
| `917_mrms_precipitation_qpe.ipynb` | MRMS QPE rain-on-grid workflow |

## Key Takeaways

### WPC QPF Characteristics

- **2.5 km CONUS grid**, issued several times daily, publicly available from WPC FTP
- **Accumulation intervals**: 6, 24, 48, 120 hours — choose based on your forecast horizon
- **GRIB2 format** with Lambert Conformal projection; cfgrib + xarray for Python reading
- **No authentication required** — data is freely accessible at
  `https://ftp.wpc.ncep.noaa.gov/2p5km_qpf/`

### qpkit API Summary

```python
from qpkit import QPKit, QPFRequest

# 1. Build request (validates immediately via pydantic)
request = QPFRequest(interval=6, cycle_hour=0)   # interval: 6 | 24 | 48 | 120

with QPKit() as kit:
    # 2. Preview without downloading
    plan = kit.build_plan_qpf(request, output_dir)     # DownloadPlan
    plan = kit.build_plan(request, output_dir)          # generic dispatcher

    # 3. Download GRIB2 files
    result = kit.download_qpf(request, output_dir)     # DownloadResult

    # 4. Access results
    result.succeeded   # tuple[Path, ...] — files written this run
    result.skipped     # tuple[Path, ...] — already cached
    result.failed      # tuple[str, ...]  — failed URLs
    result.all_files   # sorted union of succeeded + skipped
```

### Known Limitation: No Native QPF DSS Writer

At this version of qpkit (`efbe1eb`), `kit.download_to_dss(QPFRequest(...), ...)` raises
`NotImplementedError`. The QPF product does not yet have a built-in DSS grid writer. Use the
`pydsstools` manual path demonstrated in this notebook, or wait for a future qpkit release that
adds QPF DSS support.

### Common Pitfalls

- **Cycle availability**: WPC QPF files for a given `cycle_hour` may not yet be on the FTP
  server at the time you run the notebook. If the plan has 0 items, try `cycle_hour=0` (midnight
  UTC is the most reliably available cycle).
- **Longitude convention**: WPC QPF GRIB2 uses 0–360 longitudes. Convert to -180..180 with
  `lon_180 = np.where(lon > 180, lon - 360, lon)` before geographic clipping.
- **`download_to_dss` raises NotImplementedError for QPF**: This is documented behavior — use the
  manual pydsstools write path shown in this notebook.
- **Re-running with cached files**: `download_qpf` respects already-downloaded files by default
  (`force=False`). Use `kit.download_qpf(request, output_dir, force=True)` to force re-download.

---

## Attribution

**qpkit** is open-source software by **Gyan Basyal / WEST Consultants, Inc.**

- qpkit repository: https://github.com/gyanz/qpkit
- pydsstools repository: https://github.com/gyanz/pydsstools
- License: Apache 2.0

ras-commander has no native WPC QPF capability. This notebook was written to highlight
qpkit as the recommended open-source path for QPF acquisition. Please credit Gyan Basyal
and WEST Consultants when publishing workflows based on this notebook.

**WPC QPF data source**: NOAA Weather Prediction Center
https://www.wpc.ncep.noaa.gov/qpf/qpf2.shtml

## Cleanup

In [ ]:
for d in ["example_data/qpf_6hr", "example_data/qpf_24hr"]:
    p = Path(d)
    if p.exists():
        shutil.rmtree(p)
        print(f"Cleaned up: {d}")
    else:
        print(f"Not found (nothing to clean): {d}")
print("Done!")

## Acknowledgements

This workflow is powered by **[qpkit](https://github.com/gyanz/qpkit)**, an open-source library (Apache-2.0) by **[Gyan Basyal](https://github.com/gyanz)** of WEST Consultants, Inc. qpkit builds directly on **[pydsstools](https://github.com/gyanz/pydsstools)** (also authored by Gyan Basyal), extending it with NOAA precipitation acquisition (MRMS QPE, WPC QPF, HRRR) and direct GRIB2-to-HEC-DSS grid writing.

Please see and support his work:

- **qpkit** (used in this notebook): https://github.com/gyanz/qpkit
- **Gyan Basyal** on GitHub: https://github.com/gyanz
- **pydsstools** (the HEC-DSS engine qpkit builds on): https://github.com/gyanz/pydsstools
